# EnergyAI — Phase 2 exploratory analysis

This notebook explores validated hourly electricity demand for Île-de-France. Canonical timestamps remain in UTC; local calendar analysis uses `Europe/Paris`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from energyai.eda import generate_eda_report

DATA_PATH = ROOT / 'data' / 'processed' / '2024' / 'load_hourly.parquet'
REPORT_DIR = ROOT / 'reports' / 'phase2' / 'figures'
hourly = pd.read_parquet(DATA_PATH)
hourly.head()

## Data integrity

Before interpreting patterns, confirm the UTC range, missing targets, and hourly sample counts.

In [ ]:
pd.Series({
    'rows': len(hourly),
    'first_utc': hourly['timestamp_utc'].min(),
    'last_utc': hourly['timestamp_utc'].max(),
    'missing_load': hourly['load_mw'].isna().sum(),
    'complete_hours': hourly['load_mw'].notna().sum(),
})

## Demand patterns

The report covers the yearly series, load distribution, weekday/weekend profiles, monthly seasonality, a weekday-hour heatmap, and lag autocorrelation. Every axis includes its unit and timezone interpretation.

In [ ]:
figure_paths, findings = generate_eda_report(hourly, REPORT_DIR)
display(Markdown(findings))
for figure_path in figure_paths:
    display(Image(filename=str(figure_path)))

## Limitations

The RTE source is a revised historical archive, not evidence of when each measurement was available in real time. Correlations are descriptive, not causal. UTC weekly lags also shift relative to the local clock around daylight-saving transitions.